### Seq2Seq com Attention + `TimeSeriesPreprocessor`

Treina o Seq2Seq de `01_seq2seq_attention.ipynb` no split 70/15/15 do dataset completo com variantes do `mimo_sys.preprocessors.TimeSeriesPreprocessor` e mede o erro na **escala original**. Análise em `docs/2_split_artigo_vs_dataset_completo.md`.

In [1]:
from pathlib import Path

import lightning as L
import matplotlib
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

from mimo_sys.preprocessors import TimeSeriesPreprocessor

matplotlib.use("Agg")  # silencia os plots internos do preprocessor

SEED = 42
L.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PROCESSED = Path("../../data/processed")
MODELS_DIR = Path("../../models")
FIGURES_DIR = Path("../../reports/figures/training")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_COLS = ["freq_b1", "freq_b2", "freq_b3", "vazao_entrada"]
OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]
FEATURE_COLS = INPUT_COLS + OUTPUT_COLS
N_FEATURES = len(FEATURE_COLS)
N_TARGETS = len(OUTPUT_COLS)
target_idx = [FEATURE_COLS.index(c) for c in OUTPUT_COLS]

INPUT_WINDOW = 24
OUTPUT_WINDOW = 12

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/05 17:37:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Seed set to 42


### Dados

Split cronológico 70/15/15 (`03_full_dataset_sample.py`). `y_true` são as janelas de teste de 12 passos na escala original, iguais para todas as variantes (janelas deslizantes, passo 1).

In [2]:
splits = {
    name: pd.read_parquet(DATA_PROCESSED / f"mimo_full_sample_{name}.parquet")
    for name in ("train", "val", "test")
}
full_raw = pd.concat(splits.values(), ignore_index=True)[FEATURE_COLS]
full_raw = full_raw.to_numpy()
TRAIN_END = len(splits["train"])
VAL_END = TRAIN_END + len(splits["val"])
train_raw, val_raw, test_raw = (
    full_raw[:TRAIN_END],
    full_raw[TRAIN_END:VAL_END],
    full_raw[VAL_END:],
)

starts = np.arange(VAL_END, len(full_raw) - INPUT_WINDOW - OUTPUT_WINDOW + 1)
y_true = np.stack(
    [
        full_raw[s + INPUT_WINDOW : s + INPUT_WINDOW + OUTPUT_WINDOW][
            :, target_idx
        ]
        for s in starts
    ]
)
anchors = starts + INPUT_WINDOW  # 1º passo previsto, índice da série toda
print("treino/val/teste:", train_raw.shape, val_raw.shape, test_raw.shape)
print("janelas de teste:", y_true.shape)

treino/val/teste: (1545, 7) (331, 7) (332, 7)
janelas de teste: (297, 12, 3)


### Modelo Seq2Seq com dot-attention

Encoder LSTM sobre a janela de entrada (todas as features); decoder LSTM autoregressivo com dot-attention sobre os estados do encoder, prevendo as `N_TARGETS` saídas passo a passo. Porte multivariado de `data/external/seq2seq attention.ipynb`.

In [3]:
class Encoder(nn.Module):
    def __init__(
        self, input_size: int, hidden_size: int, num_layers: int = 1
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell


class DotAttention(nn.Module):
    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(
        self, hidden: torch.Tensor, encoder_outputs: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        hidden_last_layer = hidden[-1]  # [batch, hidden]
        attn_weights = torch.bmm(
            encoder_outputs.permute(1, 0, 2),
            hidden_last_layer.unsqueeze(2),
        ).squeeze(2)
        attn_weights = F.softmax(attn_weights, dim=1)

        context = torch.bmm(
            attn_weights.unsqueeze(1),
            encoder_outputs.permute(1, 0, 2),
        ).squeeze(1)

        output = torch.cat((context, hidden_last_layer), dim=1)
        output = torch.tanh(self.fc(output))
        return output, attn_weights


class Decoder(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )
        self.attention = DotAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        x: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
        encoder_outputs: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        _, (hidden, cell) = self.lstm(x, (hidden, cell))
        attn_output, attn_weights = self.attention(hidden, encoder_outputs)
        prediction = self.fc(attn_output)
        return prediction, hidden, cell, attn_weights


class Seq2SeqAttention(nn.Module):
    """Encoder-decoder LSTM with dot-attention for MIMO forecasting.

    The decoder feeds back its own prediction (padded with zeros on
    the non-target columns, since only `output_size` of `input_size`
    features are predicted) unless teacher forcing kicks in.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(input_size, hidden_size, num_layers)
        self.decoder = Decoder(
            input_size, hidden_size, output_size, num_layers
        )
        self.output_window = output_window
        self.input_size = input_size
        self.output_size = output_size

    def forward(
        self,
        source: torch.Tensor,
        target: torch.Tensor | None = None,
        teacher_forcing_ratio: float = 0.5,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        seq_len, batch_size, _ = source.shape

        outputs = torch.zeros(
            self.output_window,
            batch_size,
            self.output_size,
            device=source.device,
        )
        attention_weights = torch.zeros(
            self.output_window, batch_size, seq_len, device=source.device
        )

        encoder_outputs, hidden, cell = self.encoder(source)
        decoder_input = source[-1].unsqueeze(0)  # [1, batch, input_size]

        for t in range(self.output_window):
            decoder_output, hidden, cell, attn_weights = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[t] = decoder_output
            attention_weights[t] = attn_weights

            use_teacher_forcing = (
                target is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            )
            if use_teacher_forcing:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = target[t]
                decoder_input = next_input.unsqueeze(0)
            else:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = decoder_output
                decoder_input = next_input.unsqueeze(0)

        return outputs, attention_weights

### LightningModule

Wrapper de treino/validação/teste com MSE loss, `torchmetrics.MeanSquaredError`, `MeanAbsoluteError` e `R2Score` (por saída) em cada etapa, `AdamW` + `ReduceLROnPlateau`.

In [4]:
class Seq2SeqLightning(L.LightningModule):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
        lr: float = 1e-3,
        teacher_forcing_ratio: float = 0.5,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.model = Seq2SeqAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            output_size=output_size,
            output_window=output_window,
            num_layers=num_layers,
        )
        self.criterion = nn.MSELoss()
        self.teacher_forcing_ratio = teacher_forcing_ratio

        self.train_mse = torchmetrics.MeanSquaredError()
        self.val_mse = torchmetrics.MeanSquaredError()
        self.test_mse = torchmetrics.MeanSquaredError()
        self.train_mae = torchmetrics.MeanAbsoluteError()
        self.val_mae = torchmetrics.MeanAbsoluteError()
        self.test_mae = torchmetrics.MeanAbsoluteError()
        self.train_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.val_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.test_r2 = torchmetrics.R2Score(multioutput="raw_values")

    def _shared_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], teacher_forcing: float
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, y = batch
        x = x.permute(1, 0, 2)  # [seq_len, batch, features]
        y = y.permute(1, 0, 2)  # [output_window, batch, targets]
        outputs, _ = self.model(x, y, teacher_forcing_ratio=teacher_forcing)
        loss = self.criterion(outputs, y)
        return loss, outputs.contiguous(), y.contiguous()

    def _log_r2(
        self,
        stage: str,
        r2_metric: torchmetrics.R2Score,
        outputs: torch.Tensor,
        y: torch.Tensor,
    ) -> None:
        # R2Score expects [N, num_outputs]; flatten output_window+batch.
        flat_outputs = outputs.reshape(-1, outputs.shape[-1])
        flat_y = y.reshape(-1, y.shape[-1])
        r2_metric(flat_outputs, flat_y)
        for name, value in zip(OUTPUT_COLS, r2_metric.compute()):
            self.log(f"{stage}_r2_{name}", value, on_epoch=True, on_step=False)

    def training_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, self.teacher_forcing_ratio)
        self.train_mse(outputs, y)
        self.train_mae(outputs, y)
        self._log_r2("train", self.train_r2, outputs, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_mse", self.train_mse, on_epoch=True, on_step=False)
        self.log("train_mae", self.train_mae, on_epoch=True, on_step=False)
        return loss

    def validation_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.val_mse(outputs, y)
        self.val_mae(outputs, y)
        self._log_r2("val", self.val_r2, outputs, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mse", self.val_mse, on_epoch=True, on_step=False)
        self.log("val_mae", self.val_mae, on_epoch=True, on_step=False)
        return loss

    def test_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.test_mse(outputs, y)
        self.test_mae(outputs, y)
        self._log_r2("test", self.test_r2, outputs, y)
        self.log("test_loss", loss)
        self.log("test_mse", self.test_mse, on_epoch=True, on_step=False)
        self.log("test_mae", self.test_mae, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self) -> dict:
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

### Variantes do preprocessor

- `V0`: só `StandardScaler`.
- `V1`: outliers (IQR, limiar 1,5) + filtro Savitzky-Golay.
- `V2`: `V1` + EWT/detrend, ajustado **só no treino**; val/teste via `transform` e `inverse_transform` extrapola a tendência.
- `V3`: `V2` ajustado na **série toda** — vaza o futuro (a tendência do teste é conhecida), só como limite superior.

O limiar padrão do IQR (0,05) corta quase tudo para dentro do intervalo interquartil; por isso 1,5 explícito.

In [5]:
PP_BASE = dict(window_size=INPUT_WINDOW, horizon=OUTPUT_WINDOW)
CLEAN = dict(remove_outliers=True, outlier_threshold=1.5, apply_filter=True)
VARIANTS = {
    "V0 normalização": dict(
        remove_outliers=False,
        apply_ewt=False,
        detrend=False,
        apply_filter=False,
    ),
    "V1 outliers + filtro": dict(apply_ewt=False, detrend=False, **CLEAN),
    "V2 EWT detrend (fit treino)": dict(apply_ewt=True, detrend=True, **CLEAN),
    "V3 EWT detrend (fit série toda)": dict(
        apply_ewt=True, detrend=True, **CLEAN
    ),
}


def build_windows(name: str):
    """Janelas (X, y) de treino/val/teste no espaço do preprocessor."""
    pp = TimeSeriesPreprocessor(**PP_BASE, **VARIANTS[name])
    if name.startswith("V3"):
        X, y, _ = pp.fit_transform(full_raw, feats=target_idx)
        n_train = TRAIN_END - INPUT_WINDOW - OUTPUT_WINDOW + 1
        n_val_end = VAL_END - INPUT_WINDOW - OUTPUT_WINDOW + 1
        parts = (
            (X[:n_train], y[:n_train]),
            (X[TRAIN_END:n_val_end], y[TRAIN_END:n_val_end]),
            (X[VAL_END:], y[VAL_END:]),
        )
    else:
        X_tr, y_tr, _ = pp.fit_transform(train_raw, feats=target_idx)
        X_va, y_va = pp.transform(val_raw)
        X_te, y_te = pp.transform(test_raw)
        parts = (
            (X_tr, y_tr),
            (X_va, y_va[:, :, target_idx]),
            (X_te, y_te[:, :, target_idx]),
        )
    return pp, parts

### Treino e avaliação

`predict_original` desfaz o preprocessor com `inverse_transform` (as saídas entram numa matriz com as 7 colunas, zeradas nas entradas) e compara com `y_true` original.

In [6]:
HIDDEN_SIZE = 64
NUM_LAYERS = 2


def to_loader(X: np.ndarray, y: np.ndarray, shuffle: bool) -> DataLoader:
    tensors = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y))
    return DataLoader(tensors, batch_size=32, shuffle=shuffle)


def train_seq2seq(parts, run_name: str) -> Seq2SeqLightning:
    L.seed_everything(SEED)
    (X_tr, y_tr), (X_va, y_va), _ = parts
    model = Seq2SeqLightning(
        input_size=N_FEATURES,
        hidden_size=HIDDEN_SIZE,
        output_size=N_TARGETS,
        output_window=OUTPUT_WINDOW,
        num_layers=NUM_LAYERS,
    )
    checkpoint = ModelCheckpoint(monitor="val_mae", mode="min", save_top_k=1)
    trainer = L.Trainer(
        max_epochs=200,
        accelerator="auto",
        callbacks=[EarlyStopping(monitor="val_loss", patience=20), checkpoint],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"variant": run_name, "seed": SEED})
        trainer.fit(
            model, to_loader(X_tr, y_tr, True), to_loader(X_va, y_va, False)
        )
    best = Seq2SeqLightning.load_from_checkpoint(checkpoint.best_model_path)
    print(run_name, "melhor época:", trainer.current_epoch)
    return best.eval().to(device)


def predict_original(
    model: Seq2SeqLightning, pp: TimeSeriesPreprocessor, X_te: np.ndarray
) -> np.ndarray:
    with torch.no_grad():
        x = torch.FloatTensor(X_te).permute(1, 0, 2).to(device)
        out, _ = model.model(x, teacher_forcing_ratio=0.0)
    pred = out.permute(1, 0, 2).cpu().numpy()  # [n, 12, 3]
    padded = np.zeros((len(pred), OUTPUT_WINDOW, N_FEATURES))
    padded[:, :, target_idx] = pred
    return pp.inverse_transform(padded, anchors)[:, :, target_idx]


def metrics_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """MSE, MAE e R² por saída (janelas achatadas)."""
    yt = y_true.reshape(-1, N_TARGETS)
    yp = y_pred.reshape(-1, N_TARGETS)
    err = yt - yp
    ss_tot = ((yt - yt.mean(axis=0)) ** 2).sum(axis=0)
    return pd.DataFrame(
        {
            "mse": (err**2).mean(axis=0),
            "mae": np.abs(err).mean(axis=0),
            "r2": 1 - (err**2).sum(axis=0) / ss_tot,
        },
        index=OUTPUT_COLS,
    )


mlflow.pytorch.autolog()
mlflow.set_experiment("mimo_sys_seq2seq_preprocessor")

<Experiment: artifact_location='/home/rwp/code/mimo_sys/notebooks/training/mlruns/4', creation_time=1791232511789, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1791232511789, lifecycle_stage='active', name='mimo_sys_seq2seq_preprocessor', tags={}, trace_location=None, workspace='default'>

### Rodar as variantes

In [7]:
rows = {}
preds = {}
persistence = np.repeat(
    full_raw[starts - 1][:, None, :][:, :, target_idx], OUTPUT_WINDOW, axis=1
)
rows["Persistência (último y)"] = metrics_table(y_true, persistence)

for name in VARIANTS:
    pp, parts = build_windows(name)
    assert len(parts[2][0]) == len(y_true), (len(parts[2][0]), len(y_true))
    model = train_seq2seq(parts, name)
    preds[name] = predict_original(model, pp, parts[2][0])
    rows[name] = metrics_table(y_true, preds[name])

results = pd.concat(rows, names=["variante", "saida"])
results.round(4)

/home/rwp/code/mimo_sys/src/mimo_sys/preprocessors/timeseries.py:470: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
Seed set to 42


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


2026/10/05 17:37:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.4.0 and 2.6.5 and may not succeed with packages outside this range."


...

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


2026/10/05 17:41:28 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/10/05 17:41:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


V3 EWT detrend (fit série toda) melhor época: 40


mse      mae      r2
variante                        saida                                         
Persistência (último y)         vazao_distribuicao  1368.1131  30.8139 -1.1836
                                nivel_res              0.0919   0.2442 -0.7863
                                pressao               31.2891   4.5656 -1.1386
V0 normalização                 vazao_distribuicao    72.6848   6.4225  0.8840
                                nivel_res              0.0288   0.1339  0.4405
                                pressao                5.2451   1.5481  0.6415
V1 outliers + filtro            vazao_distribuicao    69.3278   6.2114  0.8893
                                nivel_res              0.0331   0.1398  0.3574
                                pressao                5.8653   1.5924  0.5991
V2 EWT detrend (fit treino)     vazao_distribuicao   138.4200   9.6622  0.7791
                                nivel_res              0.0372   0.1508  0.2764
                                pressao                8.3092   2.0608  0.4321
V3 EWT detrend (fit série toda) vazao_distribuicao    81.0125   6.7978  0.8707
                                nivel_res              0.0362   0.1456  0.2973
                                pressao                5.3370   1.5492  0.6352

In [8]:
results["r2"].unstack("saida").round(3)

saida,vazao_distribuicao,nivel_res,pressao
variante,,,
Persistência (último y),-1.184,-0.786,-1.139
V0 normalização,0.884,0.441,0.642
V1 outliers + filtro,0.889,0.357,0.599
V2 EWT detrend (fit treino),0.779,0.276,0.432
V3 EWT detrend (fit série toda),0.871,0.297,0.635


In [9]:
labels = ["Vazão de distribuição (l/s)", "Nível (m)", "Pressão (mca)"]
step = 0  # 1º passo previsto de cada janela
t = anchors + step
fig = make_subplots(rows=3, cols=1, subplot_titles=labels)
colors = ["royalblue", "seagreen", "darkorange", "purple"]
for i in range(3):
    fig.add_trace(
        go.Scatter(
            x=t,
            y=y_true[:, step, i],
            name="Medido",
            line=dict(color="black", width=1),
            legendgroup="m",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    for (name, pred), color in zip(preds.items(), colors):
        fig.add_trace(
            go.Scatter(
                x=t,
                y=pred[:, step, i],
                name=name,
                line=dict(color=color, width=1),
                legendgroup=name,
                showlegend=(i == 0),
            ),
            row=i + 1,
            col=1,
        )
fig.update_layout(
    title_text="Teste 70/15/15: 1º passo previsto (escala original)",
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()